# 03 - Distributions, Skewness, Sampling, and CLT

## Learning Objectives

You will:

- simulate common distributions,
- measure skewness quantitatively,
- run sampling experiments,
- understand why CLT is central for inference.


## What Are We Trying to Learn Here?

- Different distributions produce different data behavior.
- Skewness quantifies asymmetry.
- Sampling introduces variability.
- CLT explains why sample means often look normal.

## What Should We Expect?

- Normal distribution should appear symmetric.
- Poisson and some binomial settings can be right-skewed.
- Sample-mean distributions should become more normal as sample size grows.


## Step 1 - Setup

### Why this matters

Simulation lets us build statistical intuition safely before analyzing real datasets.


In [ ]:
# Core numerical and plotting libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Skewness statistic
from scipy.stats import skew

# Reproducibility
np.random.seed(42)

# Plot style for readability
sns.set_theme(style="whitegrid")


### Interpretation

This setup ensures all random simulations are reproducible and comparable.


## Step 2 - Generate Synthetic Distributions

### What we are doing

Creating normal, uniform, Poisson, and binomial samples.

### Why

Seeing multiple distributions side-by-side improves intuition about shape and support.


In [ ]:
# Number of simulated observations per distribution
n = 5000

# Build one DataFrame for convenient comparison
dist_df = pd.DataFrame({
    "normal": np.random.normal(loc=0, scale=1, size=n),         # continuous symmetric
    "uniform": np.random.uniform(low=-2, high=2, size=n),       # continuous flat
    "poisson": np.random.poisson(lam=3, size=n),                # discrete count
    "binomial": np.random.binomial(n=10, p=0.3, size=n)         # discrete successes
})

dist_df.head()


### How do we interpret output?

The first rows are not important themselves; they confirm data were generated correctly.
The key insight comes from visualizing full distributions.


## Step 3 - Visualize Distribution Shapes

### What to look for in each plot

- symmetry vs skewness,
- tail behavior,
- discrete vs continuous appearance.


In [ ]:
# Plot all distributions for visual comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), dist_df.columns):
    # Histogram with KDE for smooth shape impression
    sns.histplot(dist_df[col], kde=True, ax=ax)
    ax.set_title(f"{col.capitalize()} Distribution")

plt.tight_layout()
plt.show()


### Interpretation

- Normal should look bell-shaped and symmetric.
- Uniform should look relatively flat within interval.
- Poisson/binomial can show right skew depending on parameters.

### Why this matters

Choosing statistical tests often depends on distribution assumptions.


## Step 4 - Compute Skewness

### What we are doing

Quantifying asymmetry numerically.

### Why

Visual impressions can be subjective; skewness gives objective support.


In [ ]:
# Compute skewness for each simulated distribution
skewness_values = dist_df.apply(lambda col: skew(col, bias=False)).to_frame(name="skewness")
skewness_values


### Interpretation

- near 0 -> approximately symmetric,
- positive -> longer right tail,
- negative -> longer left tail.

### Common mistake students make

Treating small nonzero skewness as major asymmetry without considering sample size/noise.


## Conceptual Question

**Q:** Can a distribution look almost symmetric but still have small nonzero skewness?  
**A:** Yes. Finite samples can produce slight asymmetry even for symmetric populations.


## Step 5 - Sampling from a Skewed Population

### What we are doing

Using an exponential population (strong right skew).

### Why

CLT is most intuitive when population is clearly non-normal.


In [ ]:
# Large skewed population for sampling experiments
population = np.random.exponential(scale=2, size=200000)

# Visualize population shape
plt.figure(figsize=(7, 4))
sns.histplot(population, kde=True, color="tomato")
plt.title("Population Distribution (Exponential, Right-Skewed)")
plt.show()


### Interpretation

Population is clearly skewed right. This sets up a strong CLT demonstration.


## Step 6 - CLT Demonstration with Different Sample Sizes

### What should we expect before running?

- n=5: sample means still somewhat skewed,
- n=30: closer to normal,
- n=100: even more normal and tighter.


In [ ]:
# Helper function: collect repeated sample means
def sample_means(pop, sample_size, n_samples=2000):
    means = []
    for _ in range(n_samples):
        # Bootstrap-like sampling with replacement
        sample = np.random.choice(pop, size=sample_size, replace=True)
        means.append(np.mean(sample))
    return np.array(means)

# Sampling distributions for three sample sizes
means_n5 = sample_means(population, 5)
means_n30 = sample_means(population, 30)
means_n100 = sample_means(population, 100)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, arr, n_value in zip(axes, [means_n5, means_n30, means_n100], [5, 30, 100]):
    sns.histplot(arr, kde=True, ax=ax, color="slateblue")
    ax.set_title(f"Sampling Distribution of Mean (n={n_value})")
plt.tight_layout()
plt.show()


### How do we interpret this result?

As sample size grows:

- distribution of means becomes more bell-shaped,
- spread of sample means decreases,
- estimates become more stable.

### Why this matters

This is the practical reason many inferential procedures use mean-based statistics.


## Step 7 - Quantify CLT Behavior

### What we are doing

Comparing mean and standard deviation of sample-mean distributions.


In [ ]:
# Summarize CLT effect numerically
summary_clt = pd.DataFrame({
    "sample_size": [5, 30, 100],
    "mean_of_sample_means": [means_n5.mean(), means_n30.mean(), means_n100.mean()],
    "std_of_sample_means": [means_n5.std(), means_n30.std(), means_n100.std()]
})
summary_clt


### Interpretation

- Means stay near the population mean.
- Standard deviation of sample means shrinks as n increases.

### Common mistake students make

Thinking CLT makes the original data normal. It concerns the **sampling distribution of the mean**, not the raw data.


## Key Takeaways

- Distribution shape affects statistics and interpretation.
- Skewness gives a quantitative asymmetry check.
- Sampling variability is expected and measurable.
- CLT explains why larger samples improve stability of mean-based inference.
